# LeetCode 3038 — Maximum Number of Operations With the Same Score I bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-operations-with-the-same-score-i/ · Độ khó: **Easy** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def maxOperations(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lặp lại thao tác **xóa hai phần tử đầu mảng** (điểm của thao tác là **tổng hai số đó**); yêu cầu **mọi thao tác phải có cùng điểm**; tìm **số thao tác nhiều nhất**.
2. Nhận ra bài thực chất là **chia mảng thành các cặp liên tiếp `(nums[0], nums[1])`, `(nums[2], nums[3])`, ...** và đếm **dãy cặp đầu tiên có tổng bằng nhau** (dừng ngay khi có cặp lệch).
3. Biết **duyệt theo bước `2`** (`range(0, n - 1, 2)`), ghép cặp bằng `zip(nums[::2], nums[1::2])`, và dùng `itertools.takewhile`.
4. Đi từ cách **mô phỏng bằng cắt lát/xóa đầu mảng** → **duyệt chỉ số theo bước 2, không sửa mảng** → **`zip` + `takewhile`** → **danh sách tổng từng cặp rồi tìm vị trí lệch đầu tiên**.
5. Tránh các lỗi kinh điển: **không dừng** sau khi có cặp lệch (đếm cả các cặp về sau trùng điểm), dùng cặp **chồng lấn** (bước `1`), `IndexError` khi độ dài lẻ, và chỉ trả `n // 2`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn có thể thực hiện thao tác sau **trong khi `nums` còn ít nhất `2` phần tử**:

- **chọn hai phần tử đầu tiên** của `nums` và **xóa chúng**.

**Điểm** của một thao tác là **tổng của hai phần tử bị xóa** trong thao tác đó.

Bạn muốn **mọi thao tác đều có cùng điểm**. Hãy trả về **số thao tác lớn nhất** có thể thực hiện (bạn được dừng bất cứ lúc nào, nhưng **mỗi thao tác đã làm phải có điểm bằng thao tác đầu tiên**).

**Ví dụ 1**

- Input: `nums = [3, 2, 1, 4, 5]`
- Output: `2`
- Giải thích: thao tác 1: xóa `3, 2` có điểm `5`, còn `[1, 4, 5]`. Thao tác 2: xóa `1, 4` có điểm `5` (bằng điểm trước), còn `[5]`. Không còn đủ `2` phần tử, dừng. Có `2` thao tác.

**Ví dụ 2**

- Input: `nums = [1, 5, 3, 3, 4, 1, 3, 2, 2, 3]`
- Output: `2`
- Giải thích: thao tác 1: xóa `1, 5` điểm `6`. Thao tác 2: xóa `3, 3` điểm `6`. Thao tác 3 sẽ xóa `4, 1` điểm `5 != 6`, nên dừng. Có `2` thao tác (các cặp về sau như `2, 3` không được xét vì phải xóa theo thứ tự).

**Ví dụ 3**

- Input: `nums = [5, 3]`
- Output: `1`
- Giải thích: chỉ có một thao tác (xóa `5, 3` điểm `8`).

**Ràng buộc:** `2 <= nums.length <= 100`; `1 <= nums[i] <= 1000`.

In [1]:
from typing import List
from itertools import takewhile
from collections import deque
import random

examples = [
    ([3, 2, 1, 4, 5], 2),
    ([1, 5, 3, 3, 4, 1, 3, 2, 2, 3], 2),
    ([5, 3], 1),
    ([1, 2, 5, 5, 2, 1], 1),
    ([4, 4, 4, 4, 4, 4], 3),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [3, 2, 1, 4, 5] -> kỳ vọng 2
nums = [1, 5, 3, 3, 4, 1, 3, 2, 2, 3] -> kỳ vọng 2
nums = [5, 3] -> kỳ vọng 1
nums = [1, 2, 5, 5, 2, 1] -> kỳ vọng 1
nums = [4, 4, 4, 4, 4, 4] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Xóa hai phần tử đầu: cắt lát `nums[2:]`

`nums[2:]` là mảng bỏ đi hai phần tử đầu. Làm lại nhiều lần ta mô phỏng đúng thao tác của đề. Cắt lát tạo **bản sao mới** mỗi lần (tốn `O(n)`).

In [2]:
nums = [3, 2, 1, 4, 5]
print(nums[:2], sum(nums[:2]), nums[2:])        # [3, 2] 5 [1, 4, 5]
nums = nums[2:]
print(nums[:2], sum(nums[:2]), nums[2:])        # [1, 4] 5 [5]

[3, 2] 5 [1, 4, 5]
[1, 4] 5 [5]


### 3.2. Các cặp liên tiếp ứng với chỉ số `0, 2, 4, ...`

Vì mỗi thao tác xóa **hai** phần tử đầu, thao tác thứ `t` (đếm từ `0`) xóa đúng hai phần tử `nums[2t]` và `nums[2t + 1]` của **mảng ban đầu**. Nên không cần xóa thật: chỉ cần xem các cặp `(nums[0], nums[1])`, `(nums[2], nums[3])`, ... Chỉ số bắt đầu chạy theo **bước `2`**: `range(0, n - 1, 2)`.

In [3]:
n = 7
print(list(range(0, n - 1, 2)))                 # [0, 2, 4] -> các cặp (0,1), (2,3), (4,5); phần tử 6 dư
n = 6
print(list(range(0, n - 1, 2)))                 # [0, 2, 4] -> các cặp (0,1), (2,3), (4,5)

[0, 2, 4]
[0, 2, 4]


### 3.3. `range(start, stop, step)` và cận `n - 1`

`range(0, n - 1, 2)` cho các chỉ số `i` sao cho `i + 1 <= n - 1`, tức `i + 1` vẫn hợp lệ. Với `n` lẻ, phần tử cuối không có cặp và **không** xuất hiện (không gây `IndexError`). Nếu viết `range(0, n, 2)` với `n` lẻ thì `i = n - 1` làm `nums[i + 1]` ngoài biên.

In [4]:
nums = [3, 2, 1, 4, 5]                          # n = 5 lẻ
for i in range(0, len(nums) - 1, 2):
    print(i, nums[i], nums[i + 1])
try:
    for i in range(0, len(nums), 2):
        _ = nums[i] + nums[i + 1]
except IndexError as e:
    print("range(0, n, 2) gây IndexError ở n lẻ:", e)

0 3 2
2 1 4
range(0, n, 2) gây IndexError ở n lẻ: list index out of range


### 3.4. Cắt lát có bước và `zip` ghép cặp

`nums[::2]` lấy các phần tử chỉ số chẵn (**phần tử đầu mỗi cặp**), `nums[1::2]` lấy các phần tử chỉ số lẻ (**phần tử sau mỗi cặp**). `zip(nums[::2], nums[1::2])` ghép đúng các cặp liên tiếp và **tự dừng** khi một bên hết, nên phần tử lẻ dư ở cuối bị bỏ.

In [5]:
nums = [3, 2, 1, 4, 5]
print(nums[::2], nums[1::2])                     # [3, 1, 5] [2, 4]
print(list(zip(nums[::2], nums[1::2])))          # [(3, 2), (1, 4)]  (số 5 dư bị bỏ)
print([a + b for a, b in zip(nums[::2], nums[1::2])])    # [5, 5]

[3, 1, 5] [2, 4]
[(3, 2), (1, 4)]
[5, 5]


### 3.5. `itertools.takewhile`

`takewhile(điều_kiện, dãy)` lấy các phần tử **từ đầu** cho đến khi điều kiện **lần đầu sai** thì **dừng hẳn** (không xét tiếp, dù về sau điều kiện lại đúng). Đúng tinh thần "dừng ngay khi có cặp lệch".

In [6]:
print(list(takewhile(lambda x: x < 5, [1, 3, 7, 2, 4])))     # [1, 3]  (dừng ở 7, bỏ qua 2, 4)
print(list(filter(lambda x: x < 5, [1, 3, 7, 2, 4])))        # [1, 3, 2, 4]  <- filter KHÔNG dừng

[1, 3]
[1, 3, 2, 4]


### 3.6. Tìm vị trí đầu tiên thỏa điều kiện bằng `next`

`next((i for i, x in enumerate(a) if điều_kiện), mặc_định)` trả về chỉ số `i` **đầu tiên** thỏa điều kiện, hoặc `mặc_định` nếu không có. Hữu ích để biết "tới chỗ nào thì lệch".

In [7]:
sums = [6, 6, 5, 5, 5]
first_diff = next((i for i, s in enumerate(sums) if s != sums[0]), len(sums))
print(first_diff)                                  # 2: phần tử đầu tiên khác sums[0]
print(next((i for i, s in enumerate([4, 4, 4]) if s != 4), 3))    # 3: không có phần tử lệch

2
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Điểm của thao tác đầu tiên **quyết định** điểm mà mọi thao tác sau phải có. Ta xóa cặp thứ nhất rồi xem các cặp kế tiếp **theo thứ tự**; **dừng ngay** khi gặp cặp có tổng khác. Với `nums = [1, 5, 3, 3, 4, 1, 3, 2, 2, 3]`:

| cặp | `nums[2t], nums[2t+1]` | tổng | bằng điểm đầu (`6`)? | số thao tác |
|---|---|---|---|---|
| 0 | `1, 5` | 6 | (điểm chuẩn) | 1 |
| 1 | `3, 3` | 6 | có | 2 |
| 2 | `4, 1` | 5 | **không: dừng** | 2 |
| 3 | `3, 2` | 5 | (không xét nữa) | |
| 4 | `2, 3` | 5 | (không xét nữa) | |

Kết quả `2`. Chú ý: **các cặp sau chỗ lệch dù có tổng `6` cũng không được tính**: phải thực hiện thao tác theo thứ tự và mọi thao tác đã làm đều phải cùng điểm, nên chỉ có thể dừng, không thể "nhảy qua" cặp lệch.

In [8]:
nums = [1, 5, 3, 3, 4, 1, 3, 2, 2, 3]
score = nums[0] + nums[1]
count = 0
for i in range(0, len(nums) - 1, 2):
    s = nums[i] + nums[i + 1]
    ok = s == score
    print(f"cặp ({nums[i]}, {nums[i + 1]}): tổng {s} {'== ' if ok else '!= '}{score} -> {'tiếp tục' if ok else 'DỪNG'}")
    if not ok:
        break
    count += 1
print("số thao tác:", count)

cặp (1, 5): tổng 6 == 6 -> tiếp tục
cặp (3, 3): tổng 6 == 6 -> tiếp tục
cặp (4, 1): tổng 5 != 6 -> DỪNG
số thao tác: 2


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng cắt lát

**Ý tưởng.** Làm đúng như đề: `score = nums[0] + nums[1]`; trong khi còn `>= 2` phần tử và tổng hai phần tử đầu bằng `score`: tăng đếm và `nums = nums[2:]`.

In [9]:
def max_operations_simulate(nums: List[int]) -> int:
    arr = list(nums)                                   # không làm hỏng mảng của người gọi
    score = arr[0] + arr[1]
    count = 0
    while len(arr) >= 2 and arr[0] + arr[1] == score:
        count += 1
        arr = arr[2:]                                  # xóa hai phần tử đầu
    return count


for nums, expected in examples:
    print(nums, "->", max_operations_simulate(nums), "(kỳ vọng", expected, ")")

[3, 2, 1, 4, 5] -> 2 (kỳ vọng 2 )
[1, 5, 3, 3, 4, 1, 3, 2, 2, 3] -> 2 (kỳ vọng 2 )
[5, 3] -> 1 (kỳ vọng 1 )
[1, 2, 5, 5, 2, 1] -> 1 (kỳ vọng 1 )
[4, 4, 4, 4, 4, 4] -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Mỗi `arr[2:]` tạo bản sao `O(n)`, lặp tối đa `n / 2` lần: **Time `O(n²)`**, **Space `O(n)`**.

**Nút thắt.** Ta xóa thật để rồi lại nhìn đầu mảng. Có thể **không sửa mảng** mà dùng **chỉ số** `i` tiến theo bước `2`.

## 6. Ý tưởng thuật toán / bước đột phá

Thao tác thứ `t` xóa `nums[2t]` và `nums[2t + 1]` của mảng gốc, nên:

- điểm chuẩn `score = nums[0] + nums[1]`;
- thao tác thứ `t` hợp lệ ⇔ `nums[2t] + nums[2t + 1] == score` và `2t + 1 < n`;
- đáp án là **số `t` liên tiếp từ `0`** thỏa điều kiện trên (đến khi lệch hoặc hết cặp).

Chỉ cần **một chỉ số `i` bước `2`**, không xóa phần tử, không bản sao: **`O(n)` thời gian, `O(1)` bộ nhớ**. Có thể diễn đạt bằng `zip` ghép cặp và `takewhile`.

## 7. Các cách tiếp theo

### Cách 2 — Duyệt chỉ số bước `2`, dừng khi lệch

**Giải thích.** `score = nums[0] + nums[1]`. Lặp `i = 0, 2, 4, ...` (`range(0, n - 1, 2)`): nếu `nums[i] + nums[i + 1] == score` thì `count += 1`, ngược lại `break`.

In [10]:
def max_operations_index(nums: List[int]) -> int:
    score = nums[0] + nums[1]
    count = 0
    for i in range(0, len(nums) - 1, 2):               # các cặp (i, i + 1)
        if nums[i] + nums[i + 1] != score:
            break                                       # dừng hẳn khi có cặp lệch
        count += 1
    return count


for nums, expected in examples:
    print(nums, "->", max_operations_index(nums), "(kỳ vọng", expected, ")")

[3, 2, 1, 4, 5] -> 2 (kỳ vọng 2 )
[1, 5, 3, 3, 4, 1, 3, 2, 2, 3] -> 2 (kỳ vọng 2 )
[5, 3] -> 1 (kỳ vọng 1 )
[1, 2, 5, 5, 2, 1] -> 1 (kỳ vọng 1 )
[4, 4, 4, 4, 4, 4] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — `zip` ghép cặp + `takewhile`

**Giải thích.** `sums = (a + b for a, b in zip(nums[::2], nums[1::2]))` sinh tổng từng cặp (mục 3.4). `takewhile(lambda s: s == score, sums)` lấy các tổng liên tiếp bằng `score` rồi dừng ở chỗ lệch đầu tiên. Đáp án là số phần tử lấy được.

In [11]:
def max_operations_takewhile(nums: List[int]) -> int:
    score = nums[0] + nums[1]
    sums = (a + b for a, b in zip(nums[::2], nums[1::2]))     # tổng từng cặp liên tiếp (generator)
    return sum(1 for _ in takewhile(lambda s: s == score, sums))


for nums, expected in examples:
    print(nums, "->", max_operations_takewhile(nums), "(kỳ vọng", expected, ")")

[3, 2, 1, 4, 5] -> 2 (kỳ vọng 2 )
[1, 5, 3, 3, 4, 1, 3, 2, 2, 3] -> 2 (kỳ vọng 2 )
[5, 3] -> 1 (kỳ vọng 1 )
[1, 2, 5, 5, 2, 1] -> 1 (kỳ vọng 1 )
[4, 4, 4, 4, 4, 4] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** do hai lát cắt `nums[::2]`, `nums[1::2]` (generator không tạo list tổng).

### Cách 4 — Danh sách tổng từng cặp, tìm vị trí lệch đầu tiên

**Giải thích.** Tính `sums = [a + b for a, b in zip(nums[::2], nums[1::2])]`. Đáp án là **chỉ số đầu tiên** mà `sums[i] != sums[0]`, hoặc `len(sums)` nếu mọi tổng đều bằng nhau. Cách này tách rõ hai việc "tính tổng từng cặp" và "tìm chỗ lệch" (dễ in ra để debug).

In [12]:
def max_operations_sums(nums: List[int]) -> int:
    sums = [a + b for a, b in zip(nums[::2], nums[1::2])]       # tổng các cặp liên tiếp
    return next((i for i, s in enumerate(sums) if s != sums[0]), len(sums))


for nums, expected in examples:
    print(nums, "->", max_operations_sums(nums), "(kỳ vọng", expected, ")")

# Quan sát danh sách tổng của ví dụ 2
nums = [1, 5, 3, 3, 4, 1, 3, 2, 2, 3]
sums = [a + b for a, b in zip(nums[::2], nums[1::2])]
print("tổng từng cặp:", sums, "-> lệch đầu tiên tại chỉ số", next((i for i, s in enumerate(sums) if s != sums[0]), len(sums)))

[3, 2, 1, 4, 5] -> 2 (kỳ vọng 2 )
[1, 5, 3, 3, 4, 1, 3, 2, 2, 3] -> 2 (kỳ vọng 2 )
[5, 3] -> 1 (kỳ vọng 1 )
[1, 2, 5, 5, 2, 1] -> 1 (kỳ vọng 1 )
[4, 4, 4, 4, 4, 4] -> 3 (kỳ vọng 3 )
tổng từng cặp: [6, 6, 5, 5, 5] -> lệch đầu tiên tại chỉ số 2


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**.

In [13]:
# Code nộp LeetCode cho Cách 2 (O(n) thời gian, O(1) bộ nhớ)
from typing import List

class Solution:
    def maxOperations(self, nums: List[int]) -> int:
        score = nums[0] + nums[1]
        count = 0
        for i in range(0, len(nums) - 1, 2):
            if nums[i] + nums[i + 1] != score:
                break
            count += 1
        return count

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 với `[2, 4, 1, 5, 3, 3, 7]` (độ dài lẻ) và (b) Cách 3 với `takewhile`.

In [14]:
def trace_index(nums):
    score = nums[0] + nums[1]
    count = 0
    print(f"nums={nums} n={len(nums)} | score = {nums[0]} + {nums[1]} = {score}")
    for i in range(0, len(nums) - 1, 2):
        s = nums[i] + nums[i + 1]
        if s != score:
            print(f"i={i}: {nums[i]} + {nums[i + 1]} = {s} != {score} -> DỪNG")
            break
        count += 1
        print(f"i={i}: {nums[i]} + {nums[i + 1]} = {s} == {score} -> count={count}")
    else:
        print("hết cặp")
    print("=> kết quả:", count)
    return count


assert trace_index([2, 4, 1, 5, 3, 3, 7]) == 3          # 6, 6, 6; số 7 dư
assert trace_index([2, 4, 1, 5, 9, 3, 7]) == 2

nums=[2, 4, 1, 5, 3, 3, 7] n=7 | score = 2 + 4 = 6
i=0: 2 + 4 = 6 == 6 -> count=1
i=2: 1 + 5 = 6 == 6 -> count=2
i=4: 3 + 3 = 6 == 6 -> count=3
hết cặp
=> kết quả: 3
nums=[2, 4, 1, 5, 9, 3, 7] n=7 | score = 2 + 4 = 6
i=0: 2 + 4 = 6 == 6 -> count=1
i=2: 1 + 5 = 6 == 6 -> count=2
i=4: 9 + 3 = 12 != 6 -> DỪNG
=> kết quả: 2


In [15]:
def trace_takewhile(nums):
    score = nums[0] + nums[1]
    pairs = list(zip(nums[::2], nums[1::2]))
    print("các cặp:", pairs, "| score =", score)
    taken = list(takewhile(lambda p: sum(p) == score, pairs))
    print("lấy được (từ đầu, tới chỗ lệch đầu tiên):", taken)
    return len(taken)


assert trace_takewhile([1, 5, 3, 3, 4, 1, 3, 2, 2, 3]) == 2

các cặp: [(1, 5), (3, 3), (4, 1), (3, 2), (2, 3)] | score = 6
lấy được (từ đầu, tới chỗ lệch đầu tiên): [(1, 5), (3, 3)]


Bảng tóm tắt (Cách 2, `[2, 4, 1, 5, 3, 3, 7]`, `score = 6`):

| `i` | cặp | tổng | `== 6`? | `count` |
|---|---|---|---|---|
| 0 | `2, 4` | 6 | có | 1 |
| 2 | `1, 5` | 6 | có | 2 |
| 4 | `3, 3` | 6 | có | 3 |
| (6) | `7` (dư) | | không đủ cặp | 3 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (duyệt chỉ số bước `2`). Đúng class `Solution`, đúng method `maxOperations(self, nums: List[int]) -> int`.

In [16]:
from typing import List

class Solution:
    def maxOperations(self, nums: List[int]) -> int:
        score = nums[0] + nums[1]                      # điểm của thao tác đầu tiên (n >= 2)
        count = 0
        for i in range(0, len(nums) - 1, 2):           # cặp (nums[i], nums[i + 1]); i + 1 luôn hợp lệ
            if nums[i] + nums[i + 1] != score:
                break                                   # cặp lệch: mọi thao tác sau không thể thực hiện
            count += 1
        return count


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.maxOperations(nums))

[3, 2, 1, 4, 5] -> 2
[1, 5, 3, 3, 4, 1, 3, 2, 2, 3] -> 2
[5, 3] -> 1
[1, 2, 5, 5, 2, 1] -> 1
[4, 4, 4, 4, 4, 4] -> 3


## 10. Vì sao thuật toán đúng

- **Thao tác thứ `t` xóa đúng cặp `(nums[2t], nums[2t + 1])`.** Mỗi thao tác bỏ hai phần tử đầu mảng hiện tại; sau `t` thao tác, hai phần tử đầu là `nums[2t], nums[2t + 1]` của mảng gốc. Nên không cần xóa thật.
- **Điểm chuẩn.** Thao tác đầu tiên luôn là xóa `nums[0], nums[1]` (không có lựa chọn khác), nên `score = nums[0] + nums[1]` cố định; thao tác này luôn thực hiện được (`n >= 2`) và có điểm bằng chính nó, nên đáp án `>= 1`.
- **Dừng ở chỗ lệch đầu tiên.** Thao tác thứ `t` chỉ thực hiện được nếu mọi thao tác trước đó đều hợp lệ và `nums[2t] + nums[2t + 1] == score`. Khi gặp cặp lệch đầu tiên, ta không thể xóa nó (sai điểm) và cũng không thể "bỏ qua" nó để xóa cặp sau (thao tác luôn xóa **hai phần tử đầu**), nên số thao tác tối đa đúng là số cặp trước chỗ lệch.
- **Hết phần tử.** Nếu mọi cặp đầy đủ đều bằng `score` thì đáp án `n // 2`; với `n` lẻ phần tử cuối dư (còn `1` phần tử, không đủ `2`).
- **Bước `2` và cận `n - 1`.** `range(0, n - 1, 2)` chỉ gồm các `i` mà `i + 1 <= n - 1`, nên không có truy cập ngoài biên.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (cắt lát `arr[2:]`):** Time `O(n²)`, Space `O(n)`.
- **Cách 2 (chỉ số bước 2):** Time `O(n)`, Space `O(1)`.
- **Cách 3 (`zip` + `takewhile`):** Time `O(n)`, Space `O(n)`.
- **Cách 4 (danh sách tổng + `next`):** Time `O(n)`, Space `O(n)`.
- Có thể phải xem mọi cặp nên `O(n)` là tối ưu cho tệ nhất. Ở đây `n <= 100`.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "simulate": max_operations_simulate,
    "index": max_operations_index,
    "takewhile": max_operations_takewhile,
    "sums": max_operations_sums,
    "submit": sol.maxOperations,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 1], 1, "n = 2 tối thiểu")
check_all([1000, 1000], 1, "n = 2, giá trị lớn nhất")
check_all([1, 2, 3], 1, "n = 3: một cặp, phần tử cuối dư")
check_all([1, 2, 3, 4], 1, "hai cặp, tổng 3 và 7: dừng ở cặp thứ hai")
check_all([1, 2, 2, 1], 2, "hai cặp cùng tổng 3")
check_all([1, 2, 2, 1, 5], 2, "n lẻ: hai cặp bằng nhau, phần tử cuối dư")
check_all([2, 3, 1, 4, 9, 9], 2, "hai cặp tổng 5 rồi cặp lệch")
check_all([1, 2, 5, 5, 2, 1], 1, "cặp sau lệch, cặp thứ ba trở lại bằng điểm đầu: vẫn dừng ở lệch")
check_all([4, 4, 4, 4, 4, 4, 4], 3, "n = 7 lẻ, mọi cặp tổng 8")
check_all([7] * 100, 50, "n = 100 toàn bằng nhau: 50 cặp")
check_all([1, 2] * 50, 50, "n = 100: mọi cặp (1, 2) tổng 3")
check_all([1, 2] * 49 + [9, 9], 49, "n = 100: 49 cặp đúng, cặp cuối lệch")
check_all([5, 5] + [1, 2] * 49, 1, "n = 100: cặp đầu có tổng khác mọi cặp sau")

PASS n = 2 tối thiểu -> 1
PASS n = 2, giá trị lớn nhất -> 1
PASS n = 3: một cặp, phần tử cuối dư -> 1
PASS hai cặp, tổng 3 và 7: dừng ở cặp thứ hai -> 1
PASS hai cặp cùng tổng 3 -> 2
PASS n lẻ: hai cặp bằng nhau, phần tử cuối dư -> 2
PASS hai cặp tổng 5 rồi cặp lệch -> 2
PASS cặp sau lệch, cặp thứ ba trở lại bằng điểm đầu: vẫn dừng ở lệch -> 1
PASS n = 7 lẻ, mọi cặp tổng 8 -> 3
PASS n = 100 toàn bằng nhau: 50 cặp -> 50
PASS n = 100: mọi cặp (1, 2) tổng 3 -> 50
PASS n = 100: 49 cặp đúng, cặp cuối lệch -> 49
PASS n = 100: cặp đầu có tổng khác mọi cặp sau -> 1


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(11)
# Tính chất 1: 1 <= đáp án <= n // 2
for _ in range(500):
    nums = [random.randint(1, 5) for _ in range(random.randint(2, 100))]
    r = sol.maxOperations(nums)
    assert 1 <= r <= len(nums) // 2
print("PASS 1 <= đáp án <= n // 2")

# Tính chất 2: đáp án = n // 2 <=> mọi cặp đầy đủ có cùng tổng
for _ in range(500):
    nums = [random.randint(1, 3) for _ in range(random.randint(2, 40))]
    sums = [nums[i] + nums[i + 1] for i in range(0, len(nums) - 1, 2)]
    assert (sol.maxOperations(nums) == len(nums) // 2) == (len(set(sums)) == 1)
print("PASS đáp án = n // 2 <=> mọi tổng cặp bằng nhau")

# Tính chất 3: thêm phần tử ở cuối không làm giảm đáp án
for _ in range(500):
    nums = [random.randint(1, 4) for _ in range(random.randint(2, 40))]
    assert sol.maxOperations(nums + [random.randint(1, 4)]) >= sol.maxOperations(nums)
print("PASS thêm phần tử ở cuối không làm giảm đáp án")

# Tính chất 4: thay đổi cặp SAU chỗ lệch đầu tiên không đổi đáp án
for _ in range(300):
    nums = [random.randint(1, 3) for _ in range(random.randint(4, 40))]
    r = sol.maxOperations(nums)
    if 2 * r + 1 < len(nums) and nums[2 * r] + nums[2 * r + 1] != nums[0] + nums[1]:
        changed = nums[:2 * r + 2] + [random.randint(1, 3) for _ in range(len(nums) - 2 * r - 2)]
        assert sol.maxOperations(changed) == r
print("PASS phần sau chỗ lệch đầu tiên không ảnh hưởng đáp án")

# Tính chất 5: đầu vào không bị sửa
nums = [3, 2, 1, 4, 5]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: không dừng khi gặp cặp lệch (đếm mọi cặp trùng điểm chuẩn)
def sai_khong_dung(nums):
    score = nums[0] + nums[1]
    return sum(1 for i in range(0, len(nums) - 1, 2) if nums[i] + nums[i + 1] == score)

assert sai_khong_dung([1, 2, 5, 5, 2, 1]) == 2 and sol.maxOperations([1, 2, 5, 5, 2, 1]) == 1
print("PASS phản ví dụ: không dừng cho", sai_khong_dung([1, 2, 5, 5, 2, 1]), "thay vì 1")

# Phản ví dụ 2: dùng cặp chồng lấn (bước 1)
def sai_chong_lan(nums):
    score = nums[0] + nums[1]
    count = 0
    for i in range(len(nums) - 1):
        if nums[i] + nums[i + 1] != score:
            break
        count += 1
    return count

assert sai_chong_lan([2, 2, 2, 2]) == 3 and sol.maxOperations([2, 2, 2, 2]) == 2
print("PASS phản ví dụ: cặp chồng lấn cho", sai_chong_lan([2, 2, 2, 2]), "thay vì 2")

PASS 1 <= đáp án <= n // 2
PASS đáp án = n // 2 <=> mọi tổng cặp bằng nhau
PASS thêm phần tử ở cuối không làm giảm đáp án
PASS phần sau chỗ lệch đầu tiên không ảnh hưởng đáp án
PASS đầu vào không bị sửa
PASS phản ví dụ: không dừng cho 2 thay vì 1
PASS phản ví dụ: cặp chồng lấn cho 3 thay vì 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(3038)
for trial in range(3000):
    n = random.randint(2, 100)
    mode = random.choice(["small", "wide", "equal_prefix"])
    if mode == "small":
        nums = [random.randint(1, 3) for _ in range(n)]
    elif mode == "wide":
        nums = [random.randint(1, 1000) for _ in range(n)]
    else:
        total = random.randint(2, 20)
        good_pairs = random.randint(1, n // 2)
        nums = []
        for _ in range(good_pairs):
            a = random.randint(1, total - 1)
            nums += [a, total - a]                          # các cặp cùng tổng `total`
        while len(nums) < n:
            nums.append(random.randint(1, 10))
    ref = max_operations_simulate(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên (có chủ ý tạo tiền tố các cặp cùng tổng) - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên (có chủ ý tạo tiền tố các cặp cùng tổng) - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không dừng khi gặp cặp lệch

In [21]:
def sai_khong_dung_2(nums):
    score = nums[0] + nums[1]
    count = 0
    for i in range(0, len(nums) - 1, 2):
        if nums[i] + nums[i + 1] == score:
            count += 1                                  # không break: cặp sau trùng điểm vẫn được đếm
    return count


print("sai  :", sai_khong_dung_2([1, 2, 5, 5, 2, 1]), "| đúng:", Solution().maxOperations([1, 2, 5, 5, 2, 1]))

sai  : 2 | đúng: 1


**Vì sao sai:** thao tác luôn xóa **hai phần tử đầu**. Khi cặp thứ hai (`5, 5`, tổng `10`) lệch điểm, ta **không thể xóa nó** và cũng **không thể bỏ qua** nó để xóa cặp thứ ba (`2, 1`, tổng `3`). Chuỗi thao tác dừng tại đó. Cần `break` khi gặp tổng khác.

### Lỗi 2: Dùng các cặp chồng lấn (bước `1`)

In [22]:
def sai_buoc_1(nums):
    score = nums[0] + nums[1]
    count = 0
    for i in range(len(nums) - 1):                      # i = 0, 1, 2, ...: cặp (0,1), (1,2), (2,3), ...
        if nums[i] + nums[i + 1] != score:
            break
        count += 1
    return count


print("sai  :", sai_buoc_1([2, 2, 2, 2]), "| đúng:", Solution().maxOperations([2, 2, 2, 2]))

sai  : 3 | đúng: 2


**Vì sao sai:** mỗi thao tác xóa **hai** phần tử nên các cặp **không chồng lấn**: `(0, 1)`, `(2, 3)`, `(4, 5)`, ... Bước phải là `2`. Với bước `1`, phần tử `nums[1]` bị dùng cho hai cặp `(0, 1)` và `(1, 2)`, đếm thừa.

### Lỗi 3: `IndexError` khi độ dài lẻ

In [23]:
def sai_range_n(nums):
    score = nums[0] + nums[1]
    count = 0
    for i in range(0, len(nums), 2):                    # n lẻ: i = n - 1 làm nums[i + 1] ngoài biên
        if nums[i] + nums[i + 1] != score:
            break
        count += 1
    return count


try:
    print(sai_range_n([2, 4, 1, 5, 3]))
except IndexError as e:
    print("IndexError:", e, "| đúng:", Solution().maxOperations([2, 4, 1, 5, 3]))

IndexError: list index out of range | đúng: 2


**Vì sao sai:** với `n` lẻ, phần tử cuối không có cặp. `range(0, n, 2)` chạm `i = n - 1` rồi truy cập `nums[n]`. Dùng `range(0, n - 1, 2)`: chỉ lấy `i` mà `i + 1` còn trong mảng.

### Lỗi 4: Trả về `n // 2`

In [24]:
def sai_n_chia_2(nums):
    return len(nums) // 2


print("sai  :", sai_n_chia_2([1, 2, 5, 5, 2, 1]), "| đúng:", Solution().maxOperations([1, 2, 5, 5, 2, 1]))

sai  : 3 | đúng: 1


**Vì sao sai:** `n // 2` là số thao tác **nếu không có ràng buộc điểm bằng nhau**. Đề đòi mọi thao tác phải cùng điểm, nên đáp án thường **nhỏ hơn**; chỉ bằng `n // 2` khi mọi cặp có cùng tổng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "lặp lại xóa **hai phần tử đầu**" → các cặp **`(2t, 2t + 1)`** của mảng gốc; không cần xóa thật.
- Mẫu nhớ: **`for i in range(0, n - 1, 2)`** với `nums[i]` và `nums[i + 1]`.
- Thấy "mọi thao tác phải có cùng giá trị, dừng được bất cứ lúc nào" → đếm **tiền tố** các thao tác cùng giá trị, **dừng ở chỗ lệch đầu tiên** (`break` / `takewhile`).
- `zip(nums[::2], nums[1::2])` ghép các cặp liền kề không chồng lấn và tự bỏ phần tử lẻ dư.
- `takewhile` dừng hẳn ở lần sai đầu tiên; `filter` thì không.
- Mô phỏng bằng `arr = arr[2:]` đúng nhưng tốn `O(n²)`; chỉ số bước `2` đạt `O(n)`.

## 15. Biến thể của bài

1. **Bản II** (Hard): mỗi thao tác chọn xóa **hai phần tử đầu**, **hai phần tử cuối** hoặc **phần tử đầu và cuối**, mọi thao tác cùng điểm; cần quy hoạch động.
2. Xóa **hai phần tử cuối** (đối xứng).
3. Trả về **cả điểm** và số thao tác.
4. Cho phép **chọn điểm bất kỳ** (không nhất thiết là điểm của cặp đầu): tối đa hóa trên mọi điểm (khi bắt buộc cặp đầu, đáp án chính là bài này).
5. Mỗi thao tác xóa **`k`** phần tử đầu (điểm = tổng `k` phần tử).

Thử biến thể 2, 3 và 5:

In [25]:
def max_operations_from_end(nums):
    rev = nums[::-1]                                   # xóa hai phần tử cuối = xóa hai đầu của mảng đảo
    return Solution().maxOperations(rev)


def score_and_count(nums):
    return nums[0] + nums[1], Solution().maxOperations(nums)


def max_operations_k(nums, k):
    if len(nums) < k:
        return 0
    score = sum(nums[:k])
    count = 0
    for i in range(0, len(nums) - k + 1, k):
        if sum(nums[i:i + k]) != score:
            break
        count += 1
    return count


print("2.", max_operations_from_end([1, 5, 3, 3, 4, 1, 3, 2, 2, 3]))     # 3: các cặp từ cuối (3,2)=5, (2,3)=5, (1,4)=5
print("3.", score_and_count([3, 2, 1, 4, 5]))                               # (5, 2)
print("5.", max_operations_k([1, 2, 3, 3, 2, 1, 4, 4, 4], 3))              # 2: 6, 6, rồi 12
assert score_and_count([3, 2, 1, 4, 5]) == (5, 2)
assert max_operations_k([1, 2, 3, 3, 2, 1, 4, 4, 4], 3) == 2
assert max_operations_k([4, 4, 4, 4], 2) == Solution().maxOperations([4, 4, 4, 4])
assert max_operations_from_end([1, 5, 3, 3, 4, 1, 3, 2, 2, 3]) == 3

2. 3
3. (5, 2)
5. 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng `arr = arr[2:]` | làm đúng như đề | `O(n²)` | `O(n)` | hiểu đề, đáp án chuẩn |
| 2. Chỉ số bước `2` + `break` | không xóa thật | `O(n)` | `O(1)` | **cách nộp bài**, tối ưu |
| 3. `zip` + `takewhile` | tổng cặp, dừng ở chỗ lệch | `O(n)` | `O(n)` | phong cách hàm, gọn |
| 4. Danh sách tổng cặp + `next` | tìm chỉ số lệch đầu tiên | `O(n)` | `O(n)` | dễ in ra debug |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1, 2, 3, 4, 5][::2]`, `[1, 2, 3, 4, 5][1::2]`, `list(range(0, 7 - 1, 2))`, `Solution().maxOperations([4, 4, 3, 5])`, `Solution().maxOperations([2, 3, 2, 3, 2, 3, 2])`.

**Bài 2 (tính tay).** Với `nums = [3, 3, 1, 5, 2, 4, 5, 1, 7]`: ghi từng cặp, tổng, so với điểm chuẩn, và cho biết số thao tác.

**Bài 3 (điền TODO).** Hoàn thành `max_operations_todo(nums)` theo Cách 2.

In [26]:
def max_operations_todo(nums):
    score = nums[0] + nums[1]
    count = 0
    # TODO 1: duyệt i = 0, 2, 4, ... bằng range(0, len(nums) - 1, 2)
    #     TODO 2: nếu nums[i] + nums[i + 1] != score thì break
    #     TODO 3: ngược lại tăng count lên 1
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_operations_sums2(nums)` theo **Cách 4** (danh sách tổng từng cặp + vị trí lệch đầu tiên) mà không nhìn code mẫu. Kiểm tra với năm ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_operations_deque(nums)` mô phỏng bằng `collections.deque` với `popleft()` (mỗi thao tác `O(1)`). (b) Viết `max_operations_last_two(nums)`: thao tác xóa **hai phần tử cuối**. (c) Viết `best_over_all_scores(nums)`: nếu thao tác **không bắt buộc bắt đầu từ cặp đầu** mà ta được chọn **bất kỳ điểm nào** và **bất kỳ chỗ bắt đầu cặp nào** (các cặp liền kề không chồng lấn, bắt đầu tại chỉ số `s`), trả về số thao tác lớn nhất; (gợi ý: thử mọi `s` và đếm tiền tố cặp cùng tổng).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 3, 5]`, `[2, 4]`, `[0, 2, 4]`, `2` (`4 + 4 = 8` và `3 + 5 = 8`), `3` (`[2, 3, 2, 3, 2, 3, 2]`: ba cặp tổng `5`, phần tử `2` cuối dư). Bài 3: vòng `for i in range(0, len(nums) - 1, 2)` với `break` và `count += 1`.
- **Mức 2 (vừa):** Bài 2: `nums = [3, 3, 1, 5, 2, 4, 5, 1, 7]`: các cặp `(3, 3) = 6`, `(1, 5) = 6`, `(2, 4) = 6`, `(5, 1) = 6`, rồi `7` dư: `4` thao tác. Bài 4: `sums = [a + b for a, b in zip(nums[::2], nums[1::2])]`; `next((i for i, s in enumerate(sums) if s != sums[0]), len(sums))`.
- **Mức 3 (sâu):** Bài 5(a): `d = deque(nums)`, `score = d[0] + d[1]`, trong khi `len(d) >= 2` và `d[0] + d[1] == score`: `d.popleft(); d.popleft(); count += 1`. 5(b): đảo `nums` (`nums[::-1]`) rồi dùng lời giải gốc. 5(c): với mỗi `s` từ `0` đến `n - 2`, tính điểm `nums[s] + nums[s + 1]` rồi đếm các cặp `(s, s+1), (s+2, s+3), ...` cùng điểm đến khi lệch; lấy `max`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert [1, 2, 3, 4, 5][::2] == [1, 3, 5] and [1, 2, 3, 4, 5][1::2] == [2, 4]
assert list(range(0, 7 - 1, 2)) == [0, 2, 4]
assert Solution().maxOperations([4, 4, 3, 5]) == 2
assert Solution().maxOperations([2, 3, 2, 3, 2, 3, 2]) == 3

# Bài 2
assert Solution().maxOperations([3, 3, 1, 5, 2, 4, 5, 1, 7]) == 4

# Bài 3
def max_operations_todo_ans(nums):
    score = nums[0] + nums[1]
    count = 0
    for i in range(0, len(nums) - 1, 2):
        if nums[i] + nums[i + 1] != score:
            break
        count += 1
    return count

for nums, expected in examples:
    assert max_operations_todo_ans(nums) == expected

# Bài 4
def max_operations_sums2(nums):
    sums = [a + b for a, b in zip(nums[::2], nums[1::2])]
    return next((i for i, s in enumerate(sums) if s != sums[0]), len(sums))

for nums, expected in examples:
    assert max_operations_sums2(nums) == expected

# Bài 5(a)
def max_operations_deque(nums):
    d = deque(nums)
    score = d[0] + d[1]
    count = 0
    while len(d) >= 2 and d[0] + d[1] == score:
        d.popleft()
        d.popleft()
        count += 1
    return count

for nums, expected in examples:
    assert max_operations_deque(nums) == expected

# Bài 5(b)
def max_operations_last_two(nums):
    d = list(nums)
    score = d[-1] + d[-2]
    count = 0
    while len(d) >= 2 and d[-1] + d[-2] == score:
        d.pop()
        d.pop()
        count += 1
    return count

assert max_operations_last_two([1, 2, 5, 5, 2, 1]) == Solution().maxOperations([1, 2, 5, 5, 2, 1][::-1])
assert max_operations_last_two([9, 9, 3, 2, 1, 4]) == 2

# Bài 5(c)
def best_over_all_scores(nums):
    best = 0
    for s in range(len(nums) - 1):
        best = max(best, Solution().maxOperations(nums[s:]))
    return best

assert best_over_all_scores([1, 5, 2, 2, 3, 1, 4, 4, 3, 1]) >= Solution().maxOperations([1, 5, 2, 2, 3, 1, 4, 4, 3, 1])
assert best_over_all_scores([9, 1, 2, 3, 3, 2]) == 2           # bắt đầu từ s = 2: (2,3) = 5 và (3,2) = 5 nên 2 thao tác
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (các cặp `(2t, 2t + 1)`, dừng ở cặp lệch đầu tiên)

```python
from typing import List

class Solution:
    def maxOperations(self, nums: List[int]) -> int:
        score = nums[0] + nums[1]
        count = 0
        for i in range(0, len(nums) - 1, 2):
            if nums[i] + nums[i + 1] != score:
                break
            count += 1
        return count
```

**Checklist**

- [ ] Điểm chuẩn là `nums[0] + nums[1]` (thao tác đầu bắt buộc)?
- [ ] Duyệt cặp **không chồng lấn** (bước `2`), không phải bước `1`?
- [ ] **Dừng (`break`)** ở cặp lệch đầu tiên (không đếm các cặp về sau)?
- [ ] Cận `range(0, n - 1, 2)` để `n` lẻ không gây `IndexError`?
- [ ] Không trả `n // 2` một cách máy móc?
- [ ] Đã thử: `n = 2`, `n` lẻ, cặp lệch ở giữa rồi cặp sau lại trùng điểm, toàn bằng nhau (`n // 2`), `n = 100`?
- [ ] Skeleton đúng `class Solution` / `def maxOperations(self, nums: List[int]) -> int`?